# Corrected bbox 기반 ResNet18 crop 데이터셋 생성

이 노트북은 수정된 `/content/drive/MyDrive/TeamProject/test_dataset/1-stage/labels`의 9-class YOLO 라벨을 사용해
ResNet18 오염도 분류용 crop을 만듭니다.

- 입력 bbox: corrected 1-stage GT label
- 출력 클래스: `clean`, `outer`, `inner`
- 비교 padding: `0.00`, `0.05`, `0.10`
- 출력 구조: `padXXX/train|val/clean|outer|inner`

중요: `2-stage/labels`는 재질 3종만 포함하므로 오염도 crop 정답에는 사용하지 않습니다.
전체 생성 전에 경로 검사, 용량 추정, 시각 검증을 먼저 수행합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from PIL import Image, ImageOps, ImageFile
from collections import Counter, defaultdict
from IPython.display import display
import csv
import io
import json
import math
import os
import random
import shutil
import time

import matplotlib.pyplot as plt

ImageFile.LOAD_TRUNCATED_IMAGES = True

# 9-class YOLO 데이터셋
SOURCE_ROOT = Path('/content/drive/MyDrive/TeamProject/test_dataset/1-stage')
OUTPUT_ROOT = Path('/content/drive/MyDrive/TeamProject/test_dataset/crops_bboxfixed')
AUDIT_DIR = OUTPUT_ROOT / 'audit'

SPLITS = ('train', 'val')
CLASS_NAMES = ('clean', 'outer', 'inner')

DIRT_MAP = {
    0: 'clean', 1: 'outer', 2: 'inner',
    3: 'clean', 4: 'outer', 5: 'inner',
    6: 'clean', 7: 'outer', 8: 'inner',
}

IMAGE_EXTENSIONS = (
    '.jpg', '.jpeg', '.png', '.bmp', '.webp',
    '.JPG', '.JPEG', '.PNG', '.BMP', '.WEBP',
)

# 시각 비교와 전체 생성 후보
PADDING_CANDIDATES = (0.00, 0.05, 0.10)
RANDOM_SEED = 42
JPEG_QUALITY = 95

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print('SOURCE_ROOT:', SOURCE_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('padding candidates:', PADDING_CANDIDATES)


## 1. 입력 데이터 및 저장 공간 검사

학습 중인 프로세스가 완전히 끝난 뒤 실행하는 것을 권장합니다.
이 셀은 파일을 생성하지 않고 경로, 파일 수, 남은 저장 공간만 확인합니다.


In [ ]:
def find_image(image_dir: Path, stem: str):
    for extension in IMAGE_EXTENSIONS:
        candidate = image_dir / f'{stem}{extension}'
        if candidate.is_file():
            return candidate
    return None


def get_disk_status(path: Path):
    usage = shutil.disk_usage(path)
    gib = 1024 ** 3
    return {
        'total_gib': usage.total / gib,
        'used_gib': usage.used / gib,
        'free_gib': usage.free / gib,
    }


input_summary = {}
all_required_paths_exist = True

for split in SPLITS:
    image_dir = SOURCE_ROOT / 'images' / split
    label_dir = SOURCE_ROOT / 'labels' / split
    images = [p for p in image_dir.iterdir() if p.suffix in IMAGE_EXTENSIONS] if image_dir.is_dir() else []
    labels = sorted(label_dir.glob('*.txt')) if label_dir.is_dir() else []

    input_summary[split] = {
        'image_dir': str(image_dir),
        'label_dir': str(label_dir),
        'images': len(images),
        'labels': len(labels),
    }

    paths_ok = image_dir.is_dir() and label_dir.is_dir()
    all_required_paths_exist &= paths_ok

    print(f'[{split}]')
    print('  image dir:', image_dir.is_dir(), image_dir)
    print('  label dir:', label_dir.is_dir(), label_dir)
    print('  images:', len(images))
    print('  labels:', len(labels))

disk_status = get_disk_status(Path('/content/drive/MyDrive/TeamProject/test_dataset'))
print('\n[/content/drive/MyDrive/TeamProject/test_dataset 저장 공간]')
for key, value in disk_status.items():
    print(f'{key}: {value:.2f}')

if not all_required_paths_exist:
    raise FileNotFoundError('입력 경로가 없습니다. 위의 False 경로를 확인하세요.')


## 2. 라벨 구조 전체 검사

YOLO 라벨 형식 `class_id x_center y_center width height`를 검사합니다.
정규화 좌표가 0~1 범위인지, 이미지와 라벨이 대응하는지, 클래스 분포가 어떤지 확인합니다.


In [ ]:
def parse_yolo_label(label_path: Path):
    objects = []
    errors = []

    with label_path.open('r', encoding='utf-8') as file:
        lines = [line.strip() for line in file if line.strip()]

    for line_number, line in enumerate(lines, start=1):
        parts = line.split()
        if len(parts) != 5:
            errors.append(f'{label_path.name}:{line_number} 열 개수 {len(parts)}')
            continue

        try:
            class_id = int(float(parts[0]))
            xc, yc, bw, bh = map(float, parts[1:])
        except ValueError:
            errors.append(f'{label_path.name}:{line_number} 숫자 변환 실패')
            continue

        if class_id not in DIRT_MAP:
            errors.append(f'{label_path.name}:{line_number} class_id={class_id}')
            continue

        if not all(math.isfinite(v) for v in (xc, yc, bw, bh)):
            errors.append(f'{label_path.name}:{line_number} NaN/Inf')
            continue

        if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < bw <= 1 and 0 < bh <= 1):
            errors.append(
                f'{label_path.name}:{line_number} 좌표 범위 오류 '
                f'{xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}'
            )
            continue

        objects.append((class_id, xc, yc, bw, bh))

    return objects, errors


validation = {}
all_label_records = {}

for split in SPLITS:
    image_dir = SOURCE_ROOT / 'images' / split
    label_dir = SOURCE_ROOT / 'labels' / split
    label_files = sorted(label_dir.glob('*.txt'))

    class_counts = Counter()
    missing_images = []
    empty_labels = []
    label_errors = []
    records = []

    for index, label_path in enumerate(label_files, start=1):
        if index % 2000 == 0:
            print(f'{split}: {index}/{len(label_files)} 검사')

        image_path = find_image(image_dir, label_path.stem)
        if image_path is None:
            missing_images.append(label_path.name)
            continue

        objects, errors = parse_yolo_label(label_path)
        label_errors.extend(errors)
        if not objects:
            empty_labels.append(label_path.name)

        for object_index, object_data in enumerate(objects):
            class_id = object_data[0]
            dirtiness = DIRT_MAP[class_id]
            class_counts[dirtiness] += 1
            records.append({
                'split': split,
                'image_path': image_path,
                'label_path': label_path,
                'object_index': object_index,
                'class_id': class_id,
                'dirtiness': dirtiness,
                'yolo_box': object_data[1:],
            })

    all_label_records[split] = records
    validation[split] = {
        'label_files': len(label_files),
        'objects': len(records),
        'class_counts': dict(class_counts),
        'missing_images': missing_images,
        'empty_labels': empty_labels,
        'label_errors': label_errors,
    }

    print(f'\n[{split}]')
    print('labels:', len(label_files))
    print('objects:', len(records))
    print('class counts:', dict(class_counts))
    print('missing images:', len(missing_images))
    print('empty labels:', len(empty_labels))
    print('label errors:', len(label_errors))

VALIDATION_OK = all(
    len(validation[split]['missing_images']) == 0
    and len(validation[split]['empty_labels']) == 0
    and len(validation[split]['label_errors']) == 0
    for split in SPLITS
)

print('\n라벨 자동검사 통과:', VALIDATION_OK)

with (AUDIT_DIR / 'source_label_validation.json').open('w', encoding='utf-8') as file:
    json.dump(validation, file, ensure_ascii=False, indent=2)

if not VALIDATION_OK:
    raise RuntimeError('라벨 검사에 실패했습니다. 전체 crop을 생성하면 안 됩니다.')


## 3. 좌표 변환 함수와 예상 저장 용량

`padding=0.05`는 bbox의 좌우에 각각 bbox 너비의 5%, 위아래에 각각 높이의 5%를 추가합니다.
따라서 가로·세로가 각각 약 1.1배가 됩니다. 이미지 경계를 넘는 부분은 잘라냅니다.


In [ ]:
def yolo_to_xyxy(xc, yc, bw, bh, image_width, image_height, padding):
    x1 = (xc - bw / 2) * image_width
    y1 = (yc - bh / 2) * image_height
    x2 = (xc + bw / 2) * image_width
    y2 = (yc + bh / 2) * image_height

    pad_x = (x2 - x1) * padding
    pad_y = (y2 - y1) * padding

    # 왼쪽/위는 floor, 오른쪽/아래는 ceil로 객체가 잘리지 않도록 처리
    x1 = max(0, math.floor(x1 - pad_x))
    y1 = max(0, math.floor(y1 - pad_y))
    x2 = min(image_width, math.ceil(x2 + pad_x))
    y2 = min(image_height, math.ceil(y2 + pad_y))

    return x1, y1, x2, y2


all_records = [
    record
    for split in SPLITS
    for record in all_label_records[split]
]

random_generator = random.Random(RANDOM_SEED)
size_samples = random_generator.sample(
    all_records,
    k=min(300, len(all_records)),
)

estimated_sizes = defaultdict(list)

for sample_index, record in enumerate(size_samples, start=1):
    with Image.open(record['image_path']) as opened_image:
        image = ImageOps.exif_transpose(opened_image).convert('RGB')
    width, height = image.size
    xc, yc, bw, bh = record['yolo_box']

    for padding in PADDING_CANDIDATES:
        box = yolo_to_xyxy(xc, yc, bw, bh, width, height, padding)
        crop = image.crop(box)
        buffer = io.BytesIO()
        crop.save(buffer, format='JPEG', quality=JPEG_QUALITY)
        estimated_sizes[padding].append(buffer.tell())

total_objects = len(all_records)
print('전체 crop 개수(패딩 하나당):', total_objects)

for padding in PADDING_CANDIDATES:
    average_bytes = sum(estimated_sizes[padding]) / len(estimated_sizes[padding])
    estimated_gib = average_bytes * total_objects / (1024 ** 3)
    print(
        f'padding={padding:.2f} | '
        f'표본 평균={average_bytes / 1024:.1f} KiB | '
        f'예상 전체={estimated_gib:.2f} GiB'
    )

print(f"\n현재 /content/drive/MyDrive/TeamProject/test_dataset 여유 공간: {get_disk_status(Path('/content/drive/MyDrive/TeamProject/test_dataset'))['free_gib']:.2f} GiB")
print('예상치는 표본 기반이므로 실제 용량과 차이가 날 수 있습니다.')


## 4. 동일 표본에서 padding 시각 비교

각 행은 같은 객체이며 열만 padding이 달라집니다.
객체가 잘리지 않는지, 배경이 과도하게 포함되지 않는지 확인합니다.


In [ ]:
visual_random = random.Random(RANDOM_SEED)
visual_records = visual_random.sample(
    all_records,
    k=min(25, len(all_records)),
)

rows = len(visual_records)
columns = len(PADDING_CANDIDATES)
figure, axes = plt.subplots(rows, columns, figsize=(15, rows * 3.2))

if rows == 1:
    axes = [axes]

for row_index, record in enumerate(visual_records):
    with Image.open(record['image_path']) as opened_image:
        image = ImageOps.exif_transpose(opened_image).convert('RGB')
    width, height = image.size
    xc, yc, bw, bh = record['yolo_box']

    for column_index, padding in enumerate(PADDING_CANDIDATES):
        box = yolo_to_xyxy(xc, yc, bw, bh, width, height, padding)
        crop = image.crop(box)
        axis = axes[row_index][column_index]
        axis.imshow(crop)
        axis.axis('off')
        axis.set_title(
            f"{record['dirtiness']} | pad={padding:.2f}\n"
            f"{record['image_path'].name}",
            fontsize=8,
        )

figure.suptitle('GT bbox crop padding comparison', fontsize=16)
figure.tight_layout(rect=(0, 0, 1, 0.995))

comparison_path = AUDIT_DIR / 'crop_padding_comparison_random25.png'
figure.savefig(comparison_path, dpi=160, bbox_inches='tight')
plt.show()

print('저장:', comparison_path)


## 5. 사람의 시각검증 승인 및 생성 대상 선택

바로 위의 25장 비교 이미지를 직접 확인한 뒤에만 아래 값을 `True`로 바꾸세요.

공정한 padding 실험을 위해 세 후보를 모두 생성할 수 있지만, 출력 예상 용량이 여유 공간보다 크면
하나씩 생성·학습·백업·삭제하는 방식으로 진행합니다.


In [ ]:
# 위 시각화에서 corrected bbox와 padding crop이 정상임을 사람이 확인한 뒤 변경
HUMAN_VISUAL_REVIEW_PASSED = True

# 전체 생성할 padding만 선택. 예: (0.00,) 또는 (0.00, 0.05, 0.10)
PADDINGS_TO_GENERATE = (0.00, 0.05, 0.10)

# 중단 후 다시 실행하면 이미 존재하는 정상 파일은 건너뜀
SKIP_EXISTING = True

print('human visual review passed:', HUMAN_VISUAL_REVIEW_PASSED)
print('paddings to generate:', PADDINGS_TO_GENERATE)


## 6. 전체 crop 생성

원본 이미지는 객체마다 한 번만 열고 선택한 padding crop들을 함께 만듭니다.
파일은 임시 파일로 먼저 저장한 뒤 이름을 바꾸므로, 중단 시 깨진 최종 파일이 남을 위험을 줄였습니다.

기존 출력 폴더를 자동 삭제하지 않습니다.


In [ ]:
def padding_folder_name(padding):
    return f'pad{int(round(padding * 100)):03d}'


def safe_save_jpeg(image, destination: Path, quality: int):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + '.tmp')
    image.save(temporary, format='JPEG', quality=quality)
    os.replace(temporary, destination)


if not VALIDATION_OK:
    raise RuntimeError('자동검사를 통과하지 못했습니다.')

if not HUMAN_VISUAL_REVIEW_PASSED:
    raise RuntimeError(
        '시각검증 승인 전입니다. 25장 비교 이미지를 확인한 뒤 '
        'HUMAN_VISUAL_REVIEW_PASSED=True로 바꾸세요.'
    )

invalid_paddings = [
    padding for padding in PADDINGS_TO_GENERATE
    if padding not in PADDING_CANDIDATES
]
if invalid_paddings:
    raise ValueError(f'허용되지 않은 padding: {invalid_paddings}')

for padding in PADDINGS_TO_GENERATE:
    for split in SPLITS:
        for class_name in CLASS_NAMES:
            (
                OUTPUT_ROOT
                / padding_folder_name(padding)
                / split
                / class_name
            ).mkdir(parents=True, exist_ok=True)

generation_stats = {
    padding_folder_name(padding): {
        split: {class_name: 0 for class_name in CLASS_NAMES}
        for split in SPLITS
    }
    for padding in PADDINGS_TO_GENERATE
}
skipped_existing = Counter()
generation_errors = []
manifest_rows = []

generation_start = time.time()

for split in SPLITS:
    records = all_label_records[split]
    print(f'\n[{split}] 생성 시작: {len(records)} objects')

    for record_index, record in enumerate(records, start=1):
        if record_index % 500 == 0 or record_index == len(records):
            elapsed = time.time() - generation_start
            print(f'{split}: {record_index}/{len(records)} | 경과 {elapsed / 60:.1f}분')

        try:
            with Image.open(record['image_path']) as opened_image:
                image = ImageOps.exif_transpose(opened_image).convert('RGB')
        except Exception as error:
            generation_errors.append({
                'source': str(record['image_path']),
                'error': f'image open: {error}',
            })
            continue

        width, height = image.size
        xc, yc, bw, bh = record['yolo_box']
        class_name = record['dirtiness']
        stem = record['image_path'].stem
        object_index = record['object_index']
        class_id = record['class_id']
        output_name = f'{stem}__obj{object_index:02d}__cid{class_id}.jpg'

        for padding in PADDINGS_TO_GENERATE:
            folder_name = padding_folder_name(padding)
            destination = OUTPUT_ROOT / folder_name / split / class_name / output_name
            x1, y1, x2, y2 = yolo_to_xyxy(
                xc, yc, bw, bh, width, height, padding
            )

            row = {
                'split': split,
                'class_name': class_name,
                'source_image': str(record['image_path']),
                'source_label': str(record['label_path']),
                'object_index': object_index,
                'source_class_id': class_id,
                'padding': padding,
                'x1': x1,
                'y1': y1,
                'x2': x2,
                'y2': y2,
                'crop_path': str(destination),
            }
            manifest_rows.append(row)

            if x2 <= x1 or y2 <= y1:
                generation_errors.append({
                    'source': str(record['image_path']),
                    'padding': padding,
                    'error': f'invalid crop {x1},{y1},{x2},{y2}',
                })
                continue

            if SKIP_EXISTING and destination.is_file() and destination.stat().st_size > 0:
                skipped_existing[folder_name] += 1
                generation_stats[folder_name][split][class_name] += 1
                continue

            try:
                crop = image.crop((x1, y1, x2, y2))
                safe_save_jpeg(crop, destination, JPEG_QUALITY)
                generation_stats[folder_name][split][class_name] += 1
            except Exception as error:
                generation_errors.append({
                    'source': str(record['image_path']),
                    'destination': str(destination),
                    'padding': padding,
                    'error': f'crop save: {error}',
                })

manifest_path = OUTPUT_ROOT / 'crop_manifest.csv'
if manifest_rows:
    with manifest_path.open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=manifest_rows[0].keys())
        writer.writeheader()
        writer.writerows(manifest_rows)

generation_report = {
    'source_root': str(SOURCE_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'paddings': list(PADDINGS_TO_GENERATE),
    'jpeg_quality': JPEG_QUALITY,
    'stats': generation_stats,
    'skipped_existing': dict(skipped_existing),
    'errors': generation_errors,
    'manifest': str(manifest_path),
    'elapsed_minutes': (time.time() - generation_start) / 60,
}

with (AUDIT_DIR / 'crop_generation_report.json').open('w', encoding='utf-8') as file:
    json.dump(generation_report, file, ensure_ascii=False, indent=2)

print('\n===== 생성 결과 =====')
print(json.dumps(generation_report, ensure_ascii=False, indent=2))


In [ ]:
print("생성 중 기록된 오류:", len(generation_errors))

if generation_errors:
    print(generation_errors[:10])

## 7. 생성 결과 자동검증

각 padding의 생성 파일 수가 원본 라벨에서 계산한 클래스 수와 같은지 확인하고,
무작위 파일을 실제로 열어 손상 여부를 검사합니다.


In [ ]:
expected_counts = {
    split: dict(Counter(record['dirtiness'] for record in all_label_records[split]))
    for split in SPLITS
}

final_validation = {}
validation_random = random.Random(RANDOM_SEED + 1)

for padding in PADDINGS_TO_GENERATE:
    folder_name = padding_folder_name(padding)
    padding_result = {'splits': {}, 'read_errors': []}
    all_crop_paths = []

    for split in SPLITS:
        split_counts = {}
        for class_name in CLASS_NAMES:
            class_dir = OUTPUT_ROOT / folder_name / split / class_name
            paths = sorted(class_dir.glob('*.jpg'))
            split_counts[class_name] = len(paths)
            all_crop_paths.extend(paths)

        counts_match = all(
            split_counts.get(class_name, 0) == expected_counts[split].get(class_name, 0)
            for class_name in CLASS_NAMES
        )
        padding_result['splits'][split] = {
            'expected': expected_counts[split],
            'actual': split_counts,
            'counts_match': counts_match,
        }

    verify_paths = validation_random.sample(
        all_crop_paths,
        k=min(300, len(all_crop_paths)),
    )
    for crop_path in verify_paths:
        try:
            with Image.open(crop_path) as image:
                image.verify()
        except Exception as error:
            padding_result['read_errors'].append({
                'path': str(crop_path),
                'error': str(error),
            })

    padding_result['verified_sample_count'] = len(verify_paths)
    padding_result['passed'] = (
        all(item['counts_match'] for item in padding_result['splits'].values())
        and len(padding_result['read_errors']) == 0
    )
    final_validation[folder_name] = padding_result

print(json.dumps(final_validation, ensure_ascii=False, indent=2))

ALL_GENERATED_DATA_VALID = all(
    result['passed'] for result in final_validation.values()
) and len(generation_errors) == 0

print('\n전체 생성 데이터 검증 통과:', ALL_GENERATED_DATA_VALID)

with (AUDIT_DIR / 'crop_dataset_validation.json').open('w', encoding='utf-8') as file:
    json.dump(final_validation, file, ensure_ascii=False, indent=2)


## 8. 생성된 crop 무작위 시각화

각 padding별로 실제 저장된 crop을 다시 읽어 확인합니다.


In [ ]:
for padding in PADDINGS_TO_GENERATE:
    folder_name = padding_folder_name(padding)
    candidate_paths = []

    for split in SPLITS:
        for class_name in CLASS_NAMES:
            candidate_paths.extend(
                (OUTPUT_ROOT / folder_name / split / class_name).glob('*.jpg')
            )

    selected_paths = validation_random.sample(
        candidate_paths,
        k=min(25, len(candidate_paths)),
    )

    figure, axes = plt.subplots(5, 5, figsize=(15, 15))
    axes = axes.flatten()

    for axis in axes:
        axis.axis('off')

    for axis, crop_path in zip(axes, selected_paths):
        with Image.open(crop_path) as image:
            axis.imshow(image.convert('RGB'))
        axis.set_title(
            f'{crop_path.parent.name}\n{crop_path.name}',
            fontsize=7,
        )
        axis.axis('off')

    figure.suptitle(f'{folder_name} generated crops random 25', fontsize=16)
    figure.tight_layout(rect=(0, 0, 1, 0.98))
    output_path = AUDIT_DIR / f'{folder_name}_generated_random25.png'
    figure.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    print('저장:', output_path)


## 9. 최종 상태 저장

PNG를 직접 확인한 후 `FINAL_HUMAN_REVIEW_PASSED=True`로 바꾸고 실행합니다.
이 값이 참이어야 ResNet18 학습을 시작할 수 있습니다.


In [ ]:
FINAL_HUMAN_REVIEW_PASSED = True

final_status = {
    'source_root': str(SOURCE_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'source_validation_ok': VALIDATION_OK,
    'human_padding_review_passed': HUMAN_VISUAL_REVIEW_PASSED,
    'generated_dataset_validation_ok': ALL_GENERATED_DATA_VALID,
    'final_human_review_passed': FINAL_HUMAN_REVIEW_PASSED,
    'training_allowed': (
        VALIDATION_OK
        and HUMAN_VISUAL_REVIEW_PASSED
        and ALL_GENERATED_DATA_VALID
        and FINAL_HUMAN_REVIEW_PASSED
    ),
    'paddings': list(PADDINGS_TO_GENERATE),
    'manifest': str(OUTPUT_ROOT / 'crop_manifest.csv'),
    'audit_dir': str(AUDIT_DIR),
    'disk_status_gib': get_disk_status(Path('/content/drive/MyDrive/TeamProject/test_dataset')),
}

with (AUDIT_DIR / 'final_crop_status.json').open('w', encoding='utf-8') as file:
    json.dump(final_status, file, ensure_ascii=False, indent=2)

print(json.dumps(final_status, ensure_ascii=False, indent=2))

if final_status['training_allowed']:
    print('\n크롭 데이터 검증 완료: ResNet18 학습 가능')
else:
    print('\n아직 ResNet18 학습을 시작하면 안 됩니다.')
